# FMA-Medium — fine-tuned model comparison

Loads JSON summaries from `results/runs/*.json` (mode **finetune**, dataset **FMA-Medium**).  
Shows a score table, macro-averaged F1, and confusion-matrix figures when paths exist.

**Note:** If a run failed mid-suite (e.g. CLAP before the `audios=` fix), it will be missing here until you re-run that job.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Image, Markdown
except ImportError:
    display = print
    Image = None
    Markdown = str

_cwd = Path.cwd().resolve()
REPO = Path(os.environ.get("FMA_BASE_DIR", str(_cwd)))
if _cwd.name == "notebooks" and (_cwd.parent / "results" / "runs").is_dir():
    REPO = _cwd.parent
elif not (REPO / "results" / "runs").is_dir() and (_cwd / "results" / "runs").is_dir():
    REPO = _cwd
elif not (REPO / "results" / "runs").is_dir() and (_cwd.parent / "results" / "runs").is_dir():
    REPO = _cwd.parent

RUNS_DIR = REPO / "results" / "runs"
print("Repo:", REPO)
print("Runs:", RUNS_DIR)

In [ ]:
def load_run_records(runs_dir: Path) -> list[dict]:
    rows = []
    for fp in sorted(runs_dir.glob("*.json")):
        try:
            with open(fp, encoding="utf-8") as f:
                rows.append(json.load(f))
        except Exception as e:
            print("Skip", fp, e)
    return rows


def is_fma_medium_finetune(r: dict) -> bool:
    if r.get("mode") != "finetune":
        return False
    ds = str(r.get("dataset", ""))
    return ds == "fma_medium" or ds.startswith("fma_medium_")


records = load_run_records(RUNS_DIR)
medium_ft = [r for r in records if is_fma_medium_finetune(r)]
print(f"Total JSON runs: {len(records)} | FMA-Medium finetune: {len(medium_ft)}")

In [ ]:
def record_to_row(r: dict) -> dict:
    f1s = r.get("per_class_f1") or {}
    vals = [float(v) for v in f1s.values()] if f1s else []
    macro_f1 = float(np.mean(vals)) if vals else np.nan
    cfg = r.get("config") or {}
    model = r.get("model", "")
    variant = r.get("variant", "") or ""
    label = f"{model}" + (f" ({variant})" if variant else "")
    extra = {k: r.get(k) for k in ("checkpoint", "training_log_csv", "confusion_matrix_png", "training_curves_png") if r.get(k)}
    return {
        "model": model,
        "variant": variant,
        "label": label,
        "test_accuracy": float(r.get("test_accuracy", 0) or 0),
        "test_accuracy_pct": round(float(r.get("test_accuracy", 0) or 0) * 100, 2),
        "best_val_accuracy": r.get("best_val_accuracy"),
        "macro_f1": round(macro_f1, 4) if not np.isnan(macro_f1) else np.nan,
        "epochs_run": cfg.get("epochs_run", cfg.get("epochs")),
        "timestamp": r.get("timestamp", ""),
        "dataset": r.get("dataset", ""),
        "confusion_matrix_png": r.get("confusion_matrix_png", ""),
        "training_curves_png": r.get("training_curves_png", ""),
    }


rows = []
for r in medium_ft:
    row = record_to_row(r)
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print("No FMA-Medium finetune JSON found. Check results/runs and dataset tags.")
else:
    df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values("_ts").drop_duplicates("label", keep="last").drop(columns="_ts")
    df = df.sort_values("test_accuracy", ascending=False).reset_index(drop=True)
    display_cols = [
        "label",
        "test_accuracy_pct",
        "best_val_accuracy",
        "macro_f1",
        "epochs_run",
        "timestamp",
    ]
    display(df[display_cols])

## Summary table (copy-friendly)

In [ ]:
if not df.empty:
    summary = df[["label", "test_accuracy_pct", "best_val_accuracy", "macro_f1", "epochs_run"]].copy()
    summary.columns = ["Model", "Test acc %", "Best val acc", "Macro F1", "Epochs run"]
    display(summary)
    print(summary.to_markdown(index=False))
else:
    print("(empty)")

## Test accuracy comparison

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(df))
    ax.barh(df["label"], df["test_accuracy_pct"], color="steelblue")
    ax.set_xlabel("Test accuracy (%)")
    ax.set_title("FMA-Medium fine-tuned models — test accuracy")
    ax.set_xlim(0, 100)
    for i, v in enumerate(df["test_accuracy_pct"]):
        ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=9)
    plt.tight_layout()
    out = REPO / "results" / "figures" / "fma_medium_finetune_comparison.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

## Confusion matrices (saved PNGs)
One image per run, if `confusion_matrix_png` exists in the JSON.

In [ ]:
if df.empty:
    print("No rows.")
else:
    for _, row in df.iterrows():
        p = row.get("confusion_matrix_png") or ""
        p = Path(p)
        if p.is_file():
            print(row["label"], "->", p)
            if Image is not None:
                display(Image(filename=str(p), width=720))
            else:
                print("(Install IPython to display images inline)")
        else:
            print(row["label"], "-> (no confusion matrix path or file missing)", p)

## CLAP crash fix (Laion / Microsoft)
`transformers` **ClapProcessor** expects the keyword **`audios=`**, not `audio=`.  
That mismatch caused: *"You have to specify either text or audios"*.

Re-run only the failed models, e.g.:
```bash
conda activate torch
cd ~/STUDY/NU/spring26/CS5100_FAI
python models/clap/finetune.py --mode finetune --dataset medium --variant laion \\
  --epochs 40 --early_stop_patience 5 --batch_size 8 --grad_accum 2
python models/clap/finetune.py --mode finetune --dataset medium --variant microsoft \\
  --epochs 40 --early_stop_patience 5 --batch_size 8 --grad_accum 2
python models/musicldm/finetune.py --mode finetune --dataset medium \\
  --epochs 40 --early_stop_patience 5 --batch_size 16 --grad_accum 2
```
Then re-execute this notebook to refresh the table.